# Popularity-Based Recommendation (Baseline)

This notebook implements a popularity-based recommendation system as a baseline model using **The Movies Dataset** (Kaggle - Rounak Banik):
- Overall popularity recommendations
- Genre-based popularity recommendations
- Evaluate baseline performance

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import models and evaluation from src
import sys
sys.path.append('..')
from src.models import PopularityRecommender
from src.evaluation import MetricsCalculator
from src.data import DataLoader

print("Modules loaded successfully!")

Modules loaded successfully!


# Load ratings for split (the loader reads ratings_clean.csv)
loader = DataLoader()
ratings = loader.load_ratings(clean=True)
movies_metadata = loader.load_preprocessed_data('movies_integrated.csv')

print(f"Ratings loaded: {len(ratings):,} rows")

In [2]:
from src.data.data_preprocessor import DataPreprocessor

# FIX #1 — Load from the correctly split preprocessed files
# (data_preprocessing.ipynb must have been run with method='user_based' first)
loader = DataLoader()   # DataLoader imported in cell-1
movies_metadata, train_ratings, test_ratings = loader.load_preprocessed_data()

# Enforce overlap (safety net in case files were generated with temporal split)
overlap_users = set(train_ratings['userId']) & set(test_ratings['userId'])
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)].copy()
test_ratings  = test_ratings[test_ratings['userId'].isin(overlap_users)].copy()

print(f"Overlap users:  {len(overlap_users):,}")
print(f"Train ratings:  {len(train_ratings):,}")
print(f"Test ratings:   {len(test_ratings):,}")
print(f"Train users:    {train_ratings['userId'].nunique():,}")
print(f"Train movies:   {train_ratings['movieId'].nunique():,}")

# VERIFICATION GATE #1
assert len(overlap_users) >= 95, f"Too few overlap users: {len(overlap_users)}"
print("✓ Verification gate: overlap users ≥ 95")

Overlap users:  256,107
Train ratings:  20,881,168
Test ratings:   5,109,682
Train users:    256,107


Train movies:   43,372
✓ Verification gate: overlap users ≥ 95


## Overall Popularity Recommendations

In [3]:
# Initialize PopularityRecommender from src.models
pop_recommender = PopularityRecommender(train_ratings, movies_metadata, top_n=10)

# Get general recommendations
print("General Recommendations:")
print(pop_recommender.recommend().to_string(index=False))

General Recommendations:
 movieId                           title                genre_names  n_ratings  avg_rating  weighted_score
     318        The Shawshank Redemption             [Drama, Crime]      72798    4.428116        4.416915
     858                   The Godfather             [Drama, Crime]      45935    4.338718        4.322211
      50              The Usual Suspects   [Drama, Crime, Thriller]      47712    4.299380        4.283972
     527                Schindler's List      [Drama, History, War]      54351    4.267833        4.254633
    1221          The Godfather: Part II             [Drama, Crime]      29472    4.265625        4.241550
    2959                      Fight Club                    [Drama]      48241    4.227099        4.212752
    1193 One Flew Over the Cuckoo's Nest                    [Drama]      32308    4.231274        4.209904
    2019                   Seven Samurai            [Action, Drama]      11277    4.258624        4.198036
     912    

## Genre-Based Popularity Recommendations

In [4]:
# Get genre-specific recommendations
print("Action Movie Recommendations:")
print(pop_recommender.recommend(genre=['Action']).to_string(index=False))

print("\nComedy Movie Recommendations:")
print(pop_recommender.recommend(genre=['Comedy']).to_string(index=False))

print("\nDrama Movie Recommendations:")
print(pop_recommender.recommend(genre=['Drama']).to_string(index=False))

Action Movie Recommendations:
 movieId                                             title                                             genre_names  n_ratings  avg_rating  weighted_score
    2019                                     Seven Samurai                                         [Action, Drama]      11277    4.258624        4.198036
   58559                                   The Dark Knight                        [Drama, Action, Crime, Thriller]      31779    4.180890        4.160111
    2571                                        The Matrix                               [Action, Science Fiction]      62239    4.153899        4.143451
   79132                                         Inception [Action, Thriller, Science Fiction, Mystery, Adventure]      28292    4.164658        4.141711
    1196                           The Empire Strikes Back                    [Adventure, Action, Science Fiction]      49472    4.141211        4.128252
     260                                      

## Evaluate Baseline Performance

In [5]:
# Evaluate using MetricsCalculator — consistent settings across ALL notebooks
# K=10, relevance threshold=4.0, random_state=42, n_users=100
calculator = MetricsCalculator()
metrics = calculator.evaluate_model_with_recommendations(
    pop_recommender,
    test_ratings,
    k=10,
    n_users=100,
    min_rating=4.0,       # Fixed: was missing, defaulted to None
    random_state=42,
)

print("Popularity Baseline Performance Metrics (K=10, threshold=4.0):")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

# VERIFICATION GATE #1: must evaluate ≥ 95 users
assert metrics['evaluated_users'] >= 95, \
    f"Too few users evaluated: {metrics['evaluated_users']}"
print(f"✓ Verification gate: evaluated_users={metrics['evaluated_users']} ≥ 95")

Popularity Baseline Performance Metrics (K=10, threshold=4.0):
  precision@k: 0.0644
  recall@k: 0.0640
  f1@k: 0.0642
  hit_rate: 0.0580
  evaluated_users: 100.0000
  coverage: 1.0000
  catalog_coverage: 0.0012
✓ Verification gate: evaluated_users=100 ≥ 95


## Save Baseline Model

In [6]:
# Save baseline model and metrics
pop_recommender.save_model('../models')

# Save metrics using MetricsCalculator
calculator.save_metrics(metrics, '../reports/popularity_baseline_metrics.json')

print("Baseline model saved successfully!")
print("\nSaved files:")
print("- models/popularity_recommender.pkl")
print("- models/popularity_scores.csv")
print("- reports/popularity_baseline_metrics.json")

Baseline model saved successfully!

Saved files:
- models/popularity_recommender.pkl
- models/popularity_scores.csv
- reports/popularity_baseline_metrics.json


## Summary

In [7]:
print("Popularity-Based Recommendation Baseline Summary:")
print(f"Total movies in training set: {train_ratings['movieId'].nunique()}")
print(f"Total ratings in training set: {len(train_ratings):,}")
print(f"Total users in training set: {train_ratings['userId'].nunique()}")
print(f"\nTop 10 recommended movies:")
print(pop_recommender.recommend(n=10)[['title', 'weighted_score']].to_string(index=False))
print(f"\nPerformance metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

Popularity-Based Recommendation Baseline Summary:
Total movies in training set: 43372
Total ratings in training set: 20,881,168


Total users in training set: 256107

Top 10 recommended movies:
                          title  weighted_score
       The Shawshank Redemption        4.416915
                  The Godfather        4.322211
             The Usual Suspects        4.283972
               Schindler's List        4.254633
         The Godfather: Part II        4.241550
                     Fight Club        4.212752
One Flew Over the Cuckoo's Nest        4.209904
                  Seven Samurai        4.198036
                     Casablanca        4.189650
                    Rear Window        4.187001

Performance metrics:
precision@k: 0.0644
recall@k: 0.0640
f1@k: 0.0642
hit_rate: 0.0580
evaluated_users: 100.0000
coverage: 1.0000
catalog_coverage: 0.0012
